# CoastWatch Ocean Heat Content → VirtualiZarr → Icechunk on Source Cooperative

A minimal, end-to-end example of the pattern: five daily CoastWatch OHC files become an
Icechunk repository on Source Cooperative that contains **no array bytes** — only Zarr
metadata and byte-range references back to `coastwatch.noaa.gov`.

This writes to a scratch repository, `ocean-icechunks/test-repo`, **not** to the published
archive. `ocean-heat-production-sc.ipynb` builds the real stores under `noaa-ohc/`, and the
last section here will refuse to delete anything under that prefix.

One thing this example does not show: the archive changes format partway through — files up
to 2025 day 084 are NetCDF-3, later ones are HDF5 — and the two need different parsers and
cannot share a virtual array. Only 2020 NetCDF-3 files are used below; the production
notebook handles the split.

Requires Source Cooperative **write** credentials:

```
/home/jovyan/.cargo/bin/source-coop login --duration 1d --port 8400
```

In [ ]:
# Dependencies. In the git repo: pip install -r ../requirements.txt
# Standalone (downloaded on its own), uncomment:
# !pip install -q "icechunk>=2.1" "virtualizarr>=2.4" xarray obspec-utils obstore \
#     h5netcdf kerchunk scipy boto3 requests matplotlib
#
# icechunk 2.x requires Python >= 3.12. kerchunk and scipy are needed by
# NetCDF3Parser and are easy to miss — it reaches kerchunk.netCDF3.NetCDF3ToZarr,
# which subclasses scipy's netcdf_file reader.

In [ ]:
import sys
# icechunk_utils.py lives at the repo root (one level up) in the git repo, or
# alongside this notebook when downloaded from Source Cooperative. Jupyter puts
# the notebook's directory on sys.path, so a co-located copy is importable; the
# '..' entry below covers the nested git-repo layout.
sys.path.insert(0, '..')

import time
from pathlib import Path

import boto3
import requests
import xarray as xr
import icechunk
from obstore.store import HTTPStore
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import NetCDF3Parser

from icechunk_utils import open_source_icechunk_repo, get_source_credentials

## 1. Source files

The first 5 daily files from the 2020 North Atlantic OHC archive. Files begin at
day-of-year 121; adjust `doys` if earlier days are added.

In [ ]:
year = 2020
base = f"https://coastwatch.noaa.gov/pub/socd2/coastwatch/ocean_heat/na/{year}"

doys = [121, 122, 123, 124, 125]
urls = [f"{base}/ohc_naQG3_{year}_{doy}.nc" for doy in doys]

urls

## 2. Confirm HTTP byte-range access

Virtual references are only possible if the server serves byte ranges. CoastWatch does,
but it rejects the default `python-requests` User-Agent with a 403 — a browser-like UA is
required here and for every VirtualiZarr read below.

In [ ]:
BROWSER_UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/150.0.0.0 Safari/537.36"
)

r = requests.get(
    urls[0],
    headers={"User-Agent": BROWSER_UA, "Range": "bytes=0-99"},
)

print("status:", r.status_code)
print("content-range:", r.headers.get("Content-Range"))
print("accept-ranges:", r.headers.get("Accept-Ranges"))
print("bytes returned:", len(r.content))

assert r.status_code == 206, "Server did not honor the byte-range request"
assert len(r.content) == 100

## 3. Configure the HTTPS object store and the Icechunk virtual chunk container

`base` is fixed from here on: the container's `url_prefix` is derived from it, and a
manifest written against one prefix will not resolve against another.

In [ ]:
http_store = HTTPStore(
    base,
    client_options={"user_agent": BROWSER_UA},
)

registry = ObjectStoreRegistry({base: http_store})

# NetCDF3Parser uses kerchunk/fsspec internally, so the User-Agent must be
# passed via reader_options — the registry UA is not used for the metadata read.
parser = NetCDF3Parser(reader_options={
    "storage_options": {
        "client_kwargs": {"headers": {"User-Agent": BROWSER_UA}}
    }
})

http_prefix = base + "/"

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=http_prefix,
        store=icechunk.http_store(),
    )
)

credentials = icechunk.containers_credentials({
    http_prefix: icechunk.credentials.HttpAccess
})

## 4. Open or create the Icechunk repository on Source Cooperative

Writing to `ocean-icechunks/test-repo` — a scratch repository, safe to delete and rebuild.
Credentials come from the `source-coop` CLI via `icechunk_utils.open_source_icechunk_repo`.

In [ ]:
SC_BUCKET = "ocean-icechunks"
SC_PREFIX = "test-repo"   # scratch; the published archive lives under "noaa-ohc"
GROUP = "test"

repo, storage, creds, time_left = open_source_icechunk_repo(
    bucket=SC_BUCKET,
    prefix=SC_PREFIX,
    config=config,
)

# Persist the virtual chunk container so anonymous readers pick it up without
# having to pass a config of their own.
repo.save_config()

session = repo.writable_session("main")

## 5. Virtualize and append the 5 daily files

The first file creates the group hierarchy; the rest append along `time`. Each file is a
separate commit, so an interrupted loop loses only the file in flight. A writable session
is single-use — hence the fresh `writable_session` after every commit.

In [ ]:
for i, url in enumerate(urls):
    t0 = time.perf_counter()
    print(f"[{i+1}/{len(urls)}] {Path(url).name}")

    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=["time", "latitude", "longitude"],
        decode_times=True,
    ).drop_vars(["crs"], errors="ignore")

    if i == 0:
        vds.vz.to_icechunk(session.store, group=GROUP)
    else:
        vds.vz.to_icechunk(session.store, group=GROUP, append_dim="time")

    snapshot_id = session.commit(f"Add {Path(url).name}")
    print(f"  committed {snapshot_id} in {time.perf_counter() - t0:.2f}s")

    # Refresh session after each commit
    session = repo.writable_session("main")

print("Done.")

## 6. Reopen and verify

Reopening from Source Cooperative confirms the metadata round-trips. Reading a science
variable fetches the original bytes from CoastWatch over HTTPS, which is what
`authorize_virtual_chunk_access` permits.

In [ ]:
repo2 = icechunk.Repository.open(
    storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)

ds = xr.open_zarr(
    repo2.readonly_session("main").store,
    consolidated=False,
    group=GROUP,
)

ds

In [ ]:
print(ds["ohc"].isel(time=0).values)

In [ ]:
ohc = ds["ohc"].isel(time=0).where(ds["ohc"].isel(time=0) != -999)

ohc.plot(
    x="longitude",
    y="latitude",
    figsize=(10, 5),
)

## Troubleshooting — inspect and clear the scratch repository

Both cells below act on `SC_BUCKET` / `SC_PREFIX` as set in step 4. Nothing here takes a
hardcoded bucket or prefix, so re-running the notebook with a different scratch name moves
these with it.

In [ ]:
# List every object under the scratch prefix.
source_creds, _ = get_source_credentials()

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds.get("aws_session_token"),
)

keys = []
for page in s3.get_paginator("list_objects_v2").paginate(
    Bucket=SC_BUCKET, Prefix=SC_PREFIX.strip("/") + "/"
):
    for obj in page.get("Contents", []):
        keys.append(obj["Key"])
        print(f'{obj["Size"]:>12,}  {obj["Key"]}')

print(f"\nFound {len(keys):,} objects under s3://{SC_BUCKET}/{SC_PREFIX}/")

In [ ]:
# Delete everything under the scratch prefix, so the demo can be re-run from scratch.
#
# Two guards, because this operation is unrecoverable:
#   * RUN_CLEAR is a plain variable, not a `%%script false` magic. A magic on the
#     first line is one stray keystroke from being deleted; an `if` is not.
#   * PROTECTED refuses any prefix holding a published archive, so pointing this at
#     production takes more than editing one string.
RUN_CLEAR = False          # set True to actually delete
PROTECTED = {"noaa-ohc"}   # published archives — never deletable from this notebook


def clear_prefix(bucket, prefix):
    prefix = prefix.strip("/") + "/"
    stem = prefix.split("/")[0]
    if stem in PROTECTED:
        raise RuntimeError(
            f"Refusing to delete s3://{bucket}/{prefix} — {stem!r} holds a published "
            f"archive. This notebook only clears its own scratch repository."
        )

    # Gather the keys first: deleting while paginating can skip objects.
    keys = [
        obj["Key"]
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix)
        for obj in page.get("Contents", [])
    ]
    print(f"Found {len(keys):,} objects under s3://{bucket}/{prefix}")

    # Source Cooperative's S3 gateway does not support batch DeleteObjects.
    for i, key in enumerate(keys, start=1):
        s3.delete_object(Bucket=bucket, Key=key)
        if i % 100 == 0 or i == len(keys):
            print(f"  deleted {i:,} of {len(keys):,}")

    print(f"Deleted everything under s3://{bucket}/{prefix}")


if RUN_CLEAR:
    clear_prefix(SC_BUCKET, SC_PREFIX)
else:
    print(f"RUN_CLEAR is False — nothing deleted from s3://{SC_BUCKET}/{SC_PREFIX}/")